In [1]:
import json

with open("./params.json", mode = "r", encoding = "utf-8") as f:
    data = json.load(f)
    seed_vals = data["seed_vals"]
    ensemble_root_path = data["ensemble_root_path"]
    dataset_path_test_stream_windowed = data["dataset_path"]["test"]["stream"]["windowed"]
    dataset_path_test_stream_full = data["dataset_path"]["test"]["stream"]["full"]
    dataset_path_test_reversal_windowed = data["dataset_path"]["test"]["reversal"]["windowed"]
    dataset_path_test_reversal_full = data["dataset_path"]["test"]["reversal"]["full"]
    stats_path = data["stats_path"]
    num_single_sample_timesteps = data["num_single_sample_timesteps"]
    input_window_length = data["input_window_length"]
    label_window_length = data["label_window_length"]
    valid_length = input_window_length + label_window_length
    input_features = data["input_features"]
    label_features = data["label_features"]
    extra_features = data["extra_features"]
    num_label_features = len(label_features)
    window_stride = data["window_stride"]
    num_datapoints_per_timeseries = len(
        list(
            range(
                0, num_single_sample_timesteps - (input_window_length + label_window_length) + 1, window_stride
            )
        )
    )
    # num_datapoints_per_timeseries = 1 + (num_single_sample_timesteps - (input_window_length + label_window_length) + 1) // window_stride

    seed_val = 0

    # Use for kdeplot inference
    batch_size = data["batch_size"]

In [2]:
import torch
import random
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(seed_val)
random.seed(seed_val)
np.random.seed(seed_val)

In [3]:
# from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import ipywidgets as widgets
import polars as pl

from utils.pipeline.Model import TimeSeriesHuggingFaceTransformer
# from utils.pipeline.Data import get_mean_std_respected_temporal, WindowedIterableDataset, read_stats
from utils.pipeline.Data import InferenceAnalysisDataset, WindowedDataset
from utils.pipeline.Run import forward_pass

In [4]:
# stats = pl.read_csv(stats_path)

# df_test = InferenceAnalysisDataset(
#     windowed_dataset_path = dataset_path_test_windowed,
#     full_dataset_path = dataset_path_test_full,
#     index_full_update_len = num_datapoints_per_timeseries
# )

### REVERSALS ###

stats = pl.read_csv(stats_path)

df_test = InferenceAnalysisDataset(
    windowed_dataset_path = dataset_path_test_reversal_windowed,
    full_dataset_path = dataset_path_test_reversal_full,
    index_full_update_len = num_datapoints_per_timeseries
)

FileNotFoundError: The system cannot find the path specified. (os error 3): /mnt/abahari/stats5050.csv

## Prediction

In [ ]:
model_paths = [
    f"{ensemble_root_path}/22/T5-{input_window_length}-{label_window_length}-{window_stride}-{val}.pt" for val in seed_vals
]

In [ ]:
test_loss = 0.0

num_visible_timesteps = 1000

def analysis(
        target_timeseries_idx,
        feature,
        figure_range,
        selected_window_idx,
        scroll_start,
        dom_eigenvals_show,
        ensemble_mean_show
    ):
    with torch.no_grad():
        data_idx = target_timeseries_idx * num_datapoints_per_timeseries + selected_window_idx
        window_x, window_y, x_labels, extra_full = df_test[data_idx]

        x_labels = x_labels.unsqueeze(0)
        extra_full = extra_full.unsqueeze(0)
        batch_x = window_x.unsqueeze(0).to(device)
        batch_y = window_y.unsqueeze(0).to(device)

        x = list(range(num_single_sample_timesteps))

        feature_idx = label_features.index(feature)
        feature_label = batch_y[0, :, feature_idx].cpu()

        all_preds = torch.zeros((len(model_paths),) + batch_y.shape)
        all_attns = torch.zeros((len(model_paths),) + (label_window_length, input_window_length))

        for i, path in enumerate(model_paths):
            criterion = torch.nn.L1Loss()
            model = torch.load(path, weights_only = False).to(device)
            model.eval()

            outputs = forward_pass(
                model = model,
                batch_x = batch_x,
                device = device,
                extract_attention = True
            )
            preds = outputs.logits

            loss = criterion(preds, batch_y)
            print(f"Single Test Loss: {loss.item():.6f}")

            all_preds[i, :, :, :] = preds

            all_attns[i, :, :] = model.get_average_attention_values()

            del model
            torch.cuda.empty_cache()

        feature_pred = all_preds[:, 0, :, feature_idx].cpu()
        feature_x_labels = x_labels[0, :, feature_idx]
        extra_values = extra_full[0, :, 0] / 50

        feature_label = (feature_label * stats[0, f"{feature}_std"]) + stats[0, f"{feature}_mean"]
        feature_pred = (feature_pred * stats[0, f"{feature}_std"]) + stats[0, f"{feature}_mean"]

        mean_feature_preds = feature_pred.mean(dim = 0)
        std_feature_preds = feature_pred.std(dim = 0)

        ##### Attention #####
        avg_attn_vals = all_attns.mean(dim = 0).numpy()                 # (50, 100)
        avg_in_win_attns = avg_attn_vals.mean(axis = 0)                 # (100,)
        avg_in_win_attns = avg_in_win_attns / avg_in_win_attns.max()    # Normalize [0, 1] for visualization
        #####################

        # Flip detection
        is_flip = False
        input_window_start_idx = selected_window_idx * window_stride
        if(num_single_sample_timesteps // 2 >= input_window_start_idx and num_single_sample_timesteps // 2 < input_window_start_idx + valid_length):
            is_flip = True

        # Reversal probability
        u_idx = label_features.index("u_list")
        first_timesteps = all_preds[:, 0, 0, u_idx].numpy()
        last_timesteps = all_preds[:, 0, -1, u_idx].numpy()
        count_pred_reversal = np.sum(np.sign(first_timesteps) != np.sign(last_timesteps))
        reversal_prob = (count_pred_reversal / len(model_paths)) * 100
        print(f"\nReversal Probability: {reversal_prob}\n")

        sns.set_theme(style = "whitegrid")
        fig, ax = plt.subplots(figsize = (16, 8))
        ax.set_ylim(-figure_range, figure_range)

        # Input region
        ax.axvspan(
            x[selected_window_idx * window_stride],
            x[selected_window_idx * window_stride + input_window_length - 1],
            color = "green" if is_flip else "grey",
            alpha = 0.1,
            label = "Input Near-Reversal Region" if is_flip else "Input Region"
        )

        attn_cmap = sns.color_palette("Greens", as_cmap = True) if is_flip else sns.color_palette("Greys", as_cmap = True)
        for i in range(input_window_length):
            ax.axvspan(
                x[selected_window_idx * window_stride + i],
                x[selected_window_idx * window_stride + i] + 1,
                color = attn_cmap(avg_in_win_attns[i]),
                alpha = 0.6
            )

        # Label values
        sns.scatterplot(
            x = x,
            y = feature_x_labels,
            marker = "o",
            label = f"{feature}_label (circles)",
            color = "blue",
            ax = ax
        )

        # Extra values
        if(dom_eigenvals_show):
            sns.lineplot(
                x = x,
                y = extra_values,
                label = f"{extra_features[0]}",
                color = "orange",
                alpha = 0.5,
                ax = ax
            )
            plt.fill_between(
                x = x,
                y1 = extra_values,
                color = "orange",
                alpha = 0.3
            )

        # Pred values
        if(ensemble_mean_show):
            sns.scatterplot(
                x = x[(input_window_length + selected_window_idx * window_stride):(input_window_length + selected_window_idx * window_stride + label_window_length)],
                y = mean_feature_preds,
                marker = "x",
                label = f"{feature}_pred (crosses)",
                color = "red",
                ax = ax
            )
            ax.fill_between(
                x = x[(input_window_length + selected_window_idx * window_stride):(input_window_length + selected_window_idx * window_stride + label_window_length)],
                y1 = mean_feature_preds - 2 * std_feature_preds,
                y2 = mean_feature_preds + 2 * std_feature_preds,
                color = "gray",
                alpha = 0.5,
                label = f"{feature} ± 2 std"
            )
        else:
            for i in range(len(model_paths)):
                sns.lineplot(
                    x = x[(input_window_length + selected_window_idx * window_stride):(input_window_length + selected_window_idx * window_stride + label_window_length)],
                    y = feature_pred[i, :],
                    alpha = 1,
                    linewidth = 1.5,
                    label = f"model {i + 1}",
                    ax = ax,
                    # legend = False
                )

        ax.set_xlim(scroll_start, scroll_start + num_visible_timesteps)
        ax.set_title(f"{feature} Value Ground-Truth vs. Prediction")
        ax.set_xlabel("Timesteps")
        ax.set_ylabel(feature)
        ax.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)
        
        del all_attns

        for i in range(20):                                      # First 20 predictions following input sequence 
            output_row = avg_attn_vals[i, :]
            top_k_indices = np.argsort(output_row)[::-1][:10]    # Top 10 highest attention input timesteps
            top_k_scores = output_row[top_k_indices]
            print(f"Output Timestep {input_window_length + selected_window_idx * window_stride + i + 1}")
            print(f"    Input Timesteps {top_k_indices + (selected_window_idx * window_stride + 1)}")
            print(f"    Scores {[f'{score:.5f}' for score in top_k_scores]}\n")


In [ ]:
target_timeseries_idx = widgets.IntSlider(
    value = 0,
    min = 0,
    max = 1000,
    step = 1,
    description = "Timeseries Index",
    layout = widgets.Layout(width = "500px"),
    style = {"description_width": "100px"}
)
feature = widgets.Dropdown(
    options = label_features,
    value = "u_list",
    description = "Feature",
    style = {"description_width": "100px"}
)
figure_range = widgets.FloatSlider(
    value = 1,
    min = 0.1,
    max = 50,
    step = 0.1,
    description = "Figure range",
    style = {"description_width": "100px"}
)
selected_window_idx = widgets.IntSlider(
    value = 0,
    min = 0,
    max = num_datapoints_per_timeseries - 1,
    step = 1,
    description = "Window Index",
    continuous_update = False,
    layout = widgets.Layout(width = "400px"),
    style = {"description_width": "100px"}
)
scroll_start = widgets.IntSlider(
    value = 0,
    min = 0,
    max = num_single_sample_timesteps - num_visible_timesteps,
    step = 200,
    description = "Scroll X",
    layout = widgets.Layout(width = "1000px"),
    style = {"description_width": "100px"}
)
dom_eigenvals_show = widgets.Checkbox(
    value = True,
    description = "dom_eigenvals",
    disabled = False,
    indent = True,
    style = {"description_width": "100px"}
)
ensemble_mean_show = widgets.Checkbox(
    value = True,
    description = "ensemble mean",
    disabled = False,
    indent = False,
    style = {"description_width": "100px"}
)

horizontal = widgets.HBox([feature, dom_eigenvals_show, ensemble_mean_show])
ui = widgets.VBox([figure_range, target_timeseries_idx, selected_window_idx, scroll_start, horizontal])

In [ ]:
# Custom layout: interactive_output
# Prototyping: interact

out = widgets.interactive_output(
    analysis,
    {
        "target_timeseries_idx": target_timeseries_idx,
        "feature": feature,
        "figure_range": figure_range,
        "selected_window_idx": selected_window_idx,
        "scroll_start": scroll_start,
        "dom_eigenvals_show": dom_eigenvals_show,
        "ensemble_mean_show": ensemble_mean_show
    }
)
display(ui, out)

# Reversal Analysis

In [ ]:
stats = pl.read_csv(stats_path)

df_analysis = WindowedDataset(
    dataset_path = dataset_path_test_reversal_windowed
)

data_loader_infer = DataLoader(
    df_analysis,
    batch_size = batch_size,
    shuffle = False,
)

with torch.no_grad():
    all_reversal_truths = torch.zeros((len(df_analysis),))
    all_reversal_classifications = torch.zeros((len(df_analysis), len(model_paths)))

    for batch_idx, (in_win, out_win) in enumerate(data_loader_infer):
        in_win = in_win.to(device)
        out_win = out_win.to(device)

        u_idx = label_features.index("u_list")
        u_truth = out_win[:, :, u_idx]
        u_truth = (u_truth * stats[0, "u_list_std"]) + stats[0, "u_list_mean"]
        
        all_reversal_truths[
            batch_idx * batch_size : (batch_idx + 1) * batch_size
        ] = torch.sign(u_truth[:, 0]) != torch.sign(u_truth[:, -1])

        for model_idx, path in enumerate(model_paths):
            model = torch.load(path, weights_only = False).to(device)
            model.eval()

            outputs = forward_pass(
                model = model,
                batch_x = in_win,
                device = device,
                extract_attention = False
            )
            preds = outputs.logits
            preds = preds[:, :, u_idx]
            preds = (preds * stats[0, "u_list_std"]) + stats[0, "u_list_mean"]

            all_reversal_classifications[
                batch_idx * batch_size : (batch_idx + 1) * batch_size, model_idx
            ] = torch.sign(preds[:, 0]) != torch.sign(preds[:, -1])

            del model
            torch.cuda.empty_cache()


In [ ]:
all_reversal_ensemble = all_reversal_classifications.mean(axis = 1)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

fpr, tpr, roc_thresholds = roc_curve(all_reversal_truths, all_reversal_ensemble)
auc = roc_auc_score(all_reversal_truths, all_reversal_ensemble)
print(f"AUC: {auc}")

precision, recall, pr_thresholds = precision_recall_curve(all_reversal_truths, all_reversal_ensemble)
ap_score = average_precision_score(all_reversal_truths, all_reversal_ensemble)
print(f"Average Precision: {ap_score}")

In [ ]:
# TPR: Recall TP/TP+FN, FPR = FP/FP+TN

plt.figure(figsize=(6, 6))
sns.lineplot(x = fpr, y = tpr, label = f"AUC = {auc:.3f}")
sns.lineplot(x = [0, 1], y = [0, 1], linestyle = "--", color = "gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(roc_thresholds)
print(tpr)
print(fpr)

In [ ]:
baseline = sum(all_reversal_truths) / len(all_reversal_truths)

plt.figure(figsize = (6, 6))
sns.lineplot(x = recall, y = precision, label = f"AP = {ap_score:.3f}")
plt.axhline(y = baseline, linestyle = "--", color = "gray", label = f"Baseline ({baseline:.2f})")
plt.xlabel("Recall (TPR)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (6, 6))
recall_100 = [1.,         0.8255102,  0.79761905, 0.78310658, 0.76995465, 0.75918367,
0.74693878, 0.73072562, 0.70952381, 0.68367347, 0.63537415, 0.        ]
precision_100 = [0.05263158, 0.44704365, 0.50834598, 0.54385827, 0.57081617, 0.59562355,
0.62309657, 0.64482241, 0.66794749, 0.69719043, 0.73572273, 1.        ]
recall_2 = [1.,         0.84928571, 0.82857143, 0.81428571, 0.8005102,  0.78693878,
0.77377551, 0.75714286, 0.73979592, 0.7122449,  0.67163265, 0.        ]
precision_2 = [0.05263158, 0.2717803,  0.30307554, 0.32556811, 0.34447177, 0.3612178,
0.37711359, 0.39280042, 0.41155767, 0.43480969, 0.47499459, 1.        ]
precision_ = [0.05263158, 0.38737431, 0.43968603, 0.47629594, 0.50722348, 0.53476714,
 0.56292946, 0.59356231, 0.62524371, 0.66154023, 0.73041475, 1.        ]
recall_ = [1.        , 0.81678005, 0.79387755, 0.78027211, 0.76428571, 0.74988662,
 0.73378685, 0.71712018, 0.69081633, 0.65351474, 0.57505669, 0.        ]
sns.lineplot(x = recall_100, y = precision_100, label = f"AP1 = {ap_score:.3f}")
sns.lineplot(x = recall_2, y = precision_2, label = f"AP2 = {ap_score:.3f}")
sns.lineplot(x = recall_, y = precision_, label = f"AP3 = {ap_score:.3f}")
plt.axhline(y = baseline, linestyle = "--", color = "gray", label = f"Baseline ({baseline:.2f})")
plt.xlabel("Recall (TPR)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 100 50 5 all features
# [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
# [0.05263158 0.44704365 0.50834598 0.54385827 0.57081617 0.59562355
#  0.62309657 0.64482241 0.66794749 0.69719043 0.73572273 1.        ]
# [1.         0.8255102  0.79761905 0.78310658 0.76995465 0.75918367
#  0.74693878 0.73072562 0.70952381 0.68367347 0.63537415 0.        ]

# 2 50 5 all features
# [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
# [0.05263158 0.2717803  0.30307554 0.32556811 0.34447177 0.3612178
#  0.37711359 0.39280042 0.41155767 0.43480969 0.47499459 1.        ]
# [1.         0.84928571 0.82857143 0.81428571 0.8005102  0.78693878
#  0.77377551 0.75714286 0.73979592 0.7122449  0.67163265 0.        ]

print(pr_thresholds)
print(precision)
print(recall)

ROC AUC (0.899): Ensemble is great at putting most negatives at the bottom. It feels good, but it can be misleading if you have a lot of easy negatives.

AP (0.592): When you actually try to find the reversals there is a significant amount of noise.

In [ ]:
# sns.kdeplot(
#     x = first_timestep_stds,
#     y = first_timestep_extras,
#     fill = True,
#     cmap = "Blues"
# )

# plt.plot(
#     [min(first_timestep_stds), max(first_timestep_stds)],
#     [min(first_timestep_stds), max(first_timestep_stds)],
#     color = "red",
#     linestyle = "-",
#     label = "y = x"
# )

# # sns.regplot(x = first_timestep_stds, y = first_timestep_extras, scatter = False, line_kws = {"color": "green"}, ci = None)

# slope, intercept = np.polyfit(first_timestep_stds, first_timestep_extras, 1)

# # Generate points for the fit line
# x_fit = np.linspace(0, 0.04, 100)
# y_fit = slope * x_fit + intercept

# plt.plot(
#     x_fit,
#     y_fit,
#     color = "green",
#     label = "Best Fit"    
# )

# plt.xlim(-0.02, 0.04)
# plt.ylim(-0.02, 0.04)

# plt.xlabel("STDs")
# plt.ylabel("Dominant Eigenvalues")
# plt.legend()
# plt.title("First time-step std vs dom_eigenvals")

# plt.show()